In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/unified-ptbr-fakenews/Unified_PTBR_FakeNews.csv


In [ ]:
import pandas as pd 

dataset = pd.read_csv(r"codes\datasets\Unified_PTBR_FakeNews.csv")
dataset['dataset_id'].unique()

<StringArray>
[   'COVID19BR',       'FAKEBR',     'FNEWSSET',     'FRECOGNA',
      'MUMINPT', 'CENTRALFATOS',          'FCN',       'TRE300',
   'FAKETRUEBR',     'FACTCKBR',     'BOATOSBR']
Length: 11, dtype: str

In [19]:
dataset

,idx,dataset_id,text_no_url,label
0,COVID19BR_000000,COVID19BR,"O ministro da Ciência, Tecnologia, Inovações e...",REAL
1,COVID19BR_000001,COVID19BR,Pesquisa com mais de 6.000 médicos em 30 paíse...,FAKE
2,COVID19BR_000002,COVID19BR,É com muita alegria que comunico que mais um p...,REAL
3,COVID19BR_000003,COVID19BR,Renda Brasil unificará vários programas sociai...,REAL
4,COVID19BR_000004,COVID19BR,O Secretário-Geral da OTAN Jens Stoltenberg ta...,REAL
...,...,...,...,...
45272,BOATOSBR_003422,BOATOSBR,Presidente Javier Milei discursa na abertura d...,REAL
45273,BOATOSBR_003423,BOATOSBR,Milicianos interceptados pela polícia tentavam...,REAL
45274,BOATOSBR_003424,BOATOSBR,"O líder do governo no Senado, Jaques Wagner (P...",REAL
45275,BOATOSBR_003425,BOATOSBR,Fuzis e outras armas apreendidos — Foto: Henri...,REAL


## Classic models Classification

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip cache purge
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [10]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.11.0+cu128
12.8
True
NVIDIA GeForce RTX 5060 Ti


In [9]:
!nvidia-smib

'nvidia-smib' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
!pip install numpy pandas scikit-learn transformers

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split

# ============================================================
# CONFIG
# ============================================================

RANDOM_STATE = 42
MAX_LENGTH = 128
BATCH_SIZE = 16

TEST_SIZE = 0.15
VAL_SIZE = 0.15

BASE_OUTPUT_DIR = Path("results")
SPLIT_DIR = BASE_OUTPUT_DIR / "splits"
EMB_DIR = BASE_OUTPUT_DIR / "embeddings"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)

ALL_DATASETS = [
    "COVID19BR",
    "FAKEBR",
    "FNEWSSET",
    "FRECOGNA",
    "MUMINPT",
    "CENTRALFATOS",
    "FCN",
    "TRE300",
    "FAKETRUEBR",
    "FACTCKBR",
]

MODEL_CONFIGS = {
    "bertimbau_base": {
        "hf_name": "neuralmind/bert-base-portuguese-cased",
    },
    "bertimbau_large": {
        "hf_name": "neuralmind/bert-large-portuguese-cased",
    },
    "albertina_ptbr_100m": {
        "hf_name": "PORTULAN/albertina-100m-portuguese-ptbr-encoder",
    },
    "albertina_ptbr_900m": {
        "hf_name": "PORTULAN/albertina-900m-portuguese-ptbr-encoder",
    },
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando device:", device)

assert isinstance(dataset, pd.DataFrame), "A variável `dataset` deve ser um DataFrame."


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def map_label_to_int(x):
    x_str = str(x).strip().lower()
    if x_str in {"true", "real", "verdadeiro", "1"}:
        return 1
    if x_str in {"false", "fake", "falso", "0"}:
        return 0
    return np.nan


def build_strat_key(df: pd.DataFrame) -> pd.Series:
    return df["dataset_id"].astype(str) + "__" + df["label_id"].astype(str)


def safe_train_test_split(df: pd.DataFrame, test_size: float, random_state: int):
    strat_key = build_strat_key(df)
    key_counts = strat_key.value_counts()

    if (key_counts < 2).any():
        stratify_vec = df["label_id"]
        print("[WARN] Fallback de estratificação: usando apenas label_id.")
    else:
        stratify_vec = strat_key

    return train_test_split(
        df,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify_vec,
    )


def make_splits(df_input: pd.DataFrame):
    train_val_df, test_df = safe_train_test_split(
        df_input,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )

    val_relative_size = VAL_SIZE / (1.0 - TEST_SIZE)

    train_df, val_df = safe_train_test_split(
        train_val_df,
        test_size=val_relative_size,
        random_state=RANDOM_STATE,
    )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


def encode_texts(texts, tokenizer, model, device, batch_size=32, max_length=128):
    model.eval()
    all_embs = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]

            enc = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )

            enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}

            outputs = model(**enc)
            last_hidden = outputs.last_hidden_state.float()

            mask = enc["attention_mask"].unsqueeze(-1).expand(last_hidden.size()).float()
            masked = last_hidden * mask
            summed = masked.sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            emb = summed / counts

            all_embs.append(emb.cpu().numpy())

    return np.vstack(all_embs)


def save_embedding_split(encoder_name: str, split_name: str, df_split: pd.DataFrame, emb_matrix: np.ndarray):
    encoder_dir = EMB_DIR / encoder_name
    encoder_dir.mkdir(parents=True, exist_ok=True)

    emb_path = encoder_dir / f"{split_name}_embeddings.npy"
    meta_path = encoder_dir / f"{split_name}_metadata.csv"

    np.save(emb_path, emb_matrix)

    meta_df = df_split[["idx", "dataset_id", "label_id"]].copy()
    meta_df["split"] = split_name
    meta_df["embedding_model"] = encoder_name
    meta_df.to_csv(meta_path, index=False)

    print(f"[OK] {encoder_name} | {split_name} | embeddings: {emb_path}")
    print(f"[OK] {encoder_name} | {split_name} | metadata  : {meta_path}")


# ============================================================
# PREPARA BASE
# ============================================================

df_all = dataset[dataset["dataset_id"].isin(ALL_DATASETS)].copy()
df_all["label_id"] = df_all["label"].apply(map_label_to_int)
df_all = df_all.dropna(subset=["label_id"]).copy()
df_all["label_id"] = df_all["label_id"].astype(int)

required_cols = ["idx", "dataset_id", "label_id", "text_no_url"]
missing_cols = [c for c in required_cols if c not in df_all.columns]
if missing_cols:
    raise ValueError(f"Colunas obrigatórias ausentes no dataset: {missing_cols}")

train_df, val_df, test_df = make_splits(df_all)

print("\nTamanhos finais:")
print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

# salva splits
train_df.to_csv(SPLIT_DIR / "train_split.csv", index=False)
val_df.to_csv(SPLIT_DIR / "val_split.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test_split.csv", index=False)

print("\n[OK] Splits salvos em:", SPLIT_DIR)

splits_map = {
    "train": train_df,
    "val": val_df,
    "test": test_df,
}

# ============================================================
# GERA E SALVA EMBEDDINGS
# ============================================================

for alias, cfg in MODEL_CONFIGS.items():
    model_name = cfg["hf_name"]

    print("\n" + "=" * 100)
    print(f"GERANDO EMBEDDINGS PARA: {alias} ({model_name})")
    print("=" * 100)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModel.from_pretrained(
        model_name,
        trust_remote_code=True,
        torch_dtype=torch.float32,
    ).to(device)

    model = model.float().eval()

    for split_name, df_split in splits_map.items():
        t0 = time.time()

        texts = df_split["text_no_url"].astype(str).tolist()
        emb_matrix = encode_texts(
            texts,
            tokenizer,
            model,
            device=device,
            batch_size=BATCH_SIZE,
            max_length=MAX_LENGTH,
        )

        save_embedding_split(alias, split_name, df_split, emb_matrix)

        print(
            f"[{alias} | {split_name}] shape={emb_matrix.shape} | tempo={(time.time() - t0)/60:.2f} min"
        )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n[OK] Fase 1 concluída.")
print("Embeddings salvos em:", EMB_DIR)
print("Splits salvos em    :", SPLIT_DIR)

Usando device: cuda

Tamanhos finais:
Train: 29311
Val  : 6281
Test : 6281

[OK] Splits salvos em: results\splits

GERANDO EMBEDDINGS PARA: bertimbau_base (neuralmind/bert-base-portuguese-cased)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 39985.94it/s]
BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[OK] bertimbau_base | train | embeddings: results\embeddings\bertimbau_base\train_embeddings.npy
[OK] bertimbau_base | train | metadata  : results\embeddings\bertimbau_base\train_metadata.csv
[bertimbau_base | train] shape=(29311, 768) | tempo=1.43 min
[OK] bertimbau_base | val | embeddings: results\embeddings\bertimbau_base\val_embeddings.npy
[OK] bertimbau_base | val | metadata  : results\embeddings\bertimbau_base\val_metadata.csv
[bertimbau_base | val] shape=(6281, 768) | tempo=0.32 min
[OK] bertimbau_base | test | embeddings: results\embeddings\bertimbau_base\test_embeddings.npy
[OK] bertimbau_base | test | metadata  : results\embeddings\bertimbau_base\test_metadata.csv
[bertimbau_base | test] shape=(6281, 768) | tempo=0.30 min

GERANDO EMBEDDINGS PARA: bertimbau_large (neuralmind/bert-large-portuguese-cased)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 19306.29it/s]
BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[OK] bertimbau_large | train | embeddings: results\embeddings\bertimbau_large\train_embeddings.npy
[OK] bertimbau_large | train | metadata  : results\embeddings\bertimbau_large\train_metadata.csv
[bertimbau_large | train] shape=(29311, 1024) | tempo=4.21 min
[OK] bertimbau_large | val | embeddings: results\embeddings\bertimbau_large\val_embeddings.npy
[OK] bertimbau_large | val | metadata  : results\embeddings\bertimbau_large\val_metadata.csv
[bertimbau_large | val] shape=(6281, 1024) | tempo=0.91 min
[OK] bertimbau_large | test | embeddings: results\embeddings\bertimbau_large\test_embeddings.npy
[OK] bertimbau_large | test | metadata  : results\embeddings\bertimbau_large\test_metadata.csv
[bertimbau_large | test] shape=(6281, 1024) | tempo=0.90 min

GERANDO EMBEDDINGS PARA: albertina_ptbr_100m (PORTULAN/albertina-100m-portuguese-ptbr-encoder)


Loading weights: 100%|██████████| 196/196 [00:00<00:00, 1655.14it/s]
DebertaModel LOAD REPORT from: PORTULAN/albertina-100m-portuguese-ptbr-encoder
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
deberta.embeddings.position_ids            | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[OK] albertina_ptbr_100m | train | embeddings: results\embeddings\albertina_ptbr_100m\train_embeddings.npy
[OK] albertina_ptbr_100m | train | metadata  : results\embeddings\albertina_ptbr_100m\train_metadata.csv
[albertina_ptbr_100m | train] shape=(29311, 768) | tempo=2.03 min
[OK] albertina_ptbr_100m | val | embeddings: results\embeddings\albertina_ptbr_100m\val_embeddings.npy
[OK] albertina_ptbr_100m | val | metadata  : results\embeddings\albertina_ptbr_100m\val_metadata.csv
[albertina_ptbr_100m | val] shape=(6281, 768) | tempo=0.41 min
[OK] albertina_ptbr_100m | test | embeddings: results\embeddings\albertina_ptbr_100m\test_embeddings.npy
[OK] albertina_ptbr_100m | test | metadata  : results\embeddings\albertina_ptbr_100m\test_metadata.csv
[albertina_ptbr_100m | test] shape=(6281, 768) | tempo=0.42 min

GERANDO EMBEDDINGS PARA: albertina_ptbr_900m (PORTULAN/albertina-900m-portuguese-ptbr-encoder)


Loading weights: 100%|██████████| 394/394 [00:00<00:00, 499.69it/s]
DebertaV2Model LOAD REPORT from: PORTULAN/albertina-900m-portuguese-ptbr-encoder
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
deberta.embeddings.position_ids            | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[OK] albertina_ptbr_900m | train | embeddings: results\embeddings\albertina_ptbr_900m\train_embeddings.npy
[OK] albertina_ptbr_900m | train | metadata  : results\embeddings\albertina_ptbr_900m\train_metadata.csv
[albertina_ptbr_900m | train] shape=(29311, 1536) | tempo=10.80 min
[OK] albertina_ptbr_900m | val | embeddings: results\embeddings\albertina_ptbr_900m\val_embeddings.npy
[OK] albertina_ptbr_900m | val | metadata  : results\embeddings\albertina_ptbr_900m\val_metadata.csv
[albertina_ptbr_900m | val] shape=(6281, 1536) | tempo=2.38 min
[OK] albertina_ptbr_900m | test | embeddings: results\embeddings\albertina_ptbr_900m\test_embeddings.npy
[OK] albertina_ptbr_900m | test | metadata  : results\embeddings\albertina_ptbr_900m\test_metadata.csv
[albertina_ptbr_900m | test] shape=(6281, 1536) | tempo=2.37 min

[OK] Fase 1 concluída.
Embeddings salvos em: results\embeddings
Splits salvos em    : results\splits


In [15]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

# ============================================================
# CONFIG
# ============================================================

RANDOM_STATE = 42
CV_SPLITS = 3

BASE_OUTPUT_DIR = Path("results")
EMB_DIR = BASE_OUTPUT_DIR / "embeddings"
MODEL_DIR = BASE_OUTPUT_DIR / "classical_models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CONFIGS = {
    "bertimbau_base": {
        "hf_name": "neuralmind/bert-base-portuguese-cased",
    },
    "bertimbau_large": {
        "hf_name": "neuralmind/bert-large-portuguese-cased",
    },
    "albertina_ptbr_100m": {
        "hf_name": "PORTULAN/albertina-100m-portuguese-ptbr-encoder",
    },
    "albertina_ptbr_900m": {
        "hf_name": "PORTULAN/albertina-900m-portuguese-ptbr-encoder",
    },
}

from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

CLASSICAL_MODELS = {
    "logreg": {
        "pipeline": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
            ]
        ),
        "param_grid": {
            "clf__C": [0.01, 0.1, 1.0, 10.0],
            "clf__solver": ["liblinear", "lbfgs"],
        },
    },
    "linearsvc": {
        "pipeline": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("clf", LinearSVC(random_state=RANDOM_STATE, max_iter=8000)),
            ]
        ),
        "param_grid": {
            "clf__C": [0.01, 0.1, 1.0, 10.0],
        },
    },
    "svc_rbf": {
        "pipeline": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("clf", SVC(kernel="rbf")),
            ]
        ),
        "param_grid": {
            "clf__C": [1.0, 10.0],
            "clf__gamma": ["scale"],
        },
    },
    "random_forest": {
        "pipeline": Pipeline(
            [
                ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
            ]
        ),
        "param_grid": {
            "clf__n_estimators": [200, 500],
            "clf__max_depth": [None, 20],
            "clf__min_samples_split": [2],
        },
    },
    "gradient_boosting": {
        "pipeline": Pipeline(
            [
                ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE)),
            ]
        ),
        "param_grid": {
            "clf__n_estimators": [100, 200],
            "clf__learning_rate": [0.1],
            "clf__max_depth": [3],
        },
    },
    "gaussian_nb": {
        "pipeline": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("clf", GaussianNB()),
            ]
        ),
        "param_grid": {
            "clf__var_smoothing": [1e-9, 1e-8],
        },
    },
    "knn": {
        "pipeline": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("clf", KNeighborsClassifier()),
            ]
        ),
        "param_grid": {
            "clf__n_neighbors": [3, 5, 7, 11],
            "clf__weights": ["uniform", "distance"],
            "clf__metric": ["minkowski"],
        },
    },
    "decision_tree": {
        "pipeline": Pipeline(
            [
                ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
            ]
        ),
        "param_grid": {
            "clf__max_depth": [None, 10, 20, 30],
            "clf__min_samples_split": [2, 5, 10],
            "clf__criterion": ["gini", "entropy"],
        },
    },
}

cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)


# ============================================================
# FUNÇÃO DE CARGA
# ============================================================

def load_embedding_split(encoder_name: str, split_name: str):
    encoder_dir = EMB_DIR / encoder_name
    emb_path = encoder_dir / f"{split_name}_embeddings.npy"
    meta_path = encoder_dir / f"{split_name}_metadata.csv"

    if not emb_path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {emb_path}")
    if not meta_path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {meta_path}")

    X = np.load(emb_path)
    meta = pd.read_csv(meta_path)

    return X, meta


# ============================================================
# TREINO / VAL / TEST
# ============================================================

results = []
rows_per_item = []

for encoder_name in MODEL_CONFIGS.keys():
    print("\n" + "#" * 100)
    print(f"CARREGANDO EMBEDDINGS SALVOS PARA: {encoder_name}")
    print("#" * 100)

    X_train, meta_train = load_embedding_split(encoder_name, "train")
    X_val, meta_val = load_embedding_split(encoder_name, "val")
    X_test, meta_test = load_embedding_split(encoder_name, "test")

    y_train = meta_train["label_id"].to_numpy()
    y_val = meta_val["label_id"].to_numpy()
    y_test = meta_test["label_id"].to_numpy()

    if X_train.shape[0] != len(meta_train):
        raise ValueError(f"Inconsistência em train para {encoder_name}")
    if X_val.shape[0] != len(meta_val):
        raise ValueError(f"Inconsistência em val para {encoder_name}")
    if X_test.shape[0] != len(meta_test):
        raise ValueError(f"Inconsistência em test para {encoder_name}")

    X_train_val = np.vstack([X_train, X_val])
    y_train_val = np.concatenate([y_train, y_val], axis=0)

    for clf_name, clf_cfg in CLASSICAL_MODELS.items():
        experiment_name = f"{encoder_name}__{clf_name}"

        print("\n" + "-" * 100)
        print(f"TREINANDO: {experiment_name}")
        print("-" * 100)

        grid = GridSearchCV(
            estimator=clf_cfg["pipeline"],
            param_grid=clf_cfg["param_grid"],
            scoring="f1_macro",
            cv=cv,
            n_jobs=1,
            verbose=1,
            refit=True,
        )

        grid.fit(X_train, y_train)
        best_clf = grid.best_estimator_

        print("Melhores hiperparâmetros:", grid.best_params_)
        print("Melhor CV F1-macro:", grid.best_score_)

        # validação
        y_val_pred = best_clf.predict(X_val)
        val_acc = accuracy_score(y_val, y_val_pred)
        val_f1_macro = f1_score(y_val, y_val_pred, average="macro")

        print(f"\n[VAL] {experiment_name} | Acc={val_acc:.4f} | F1-macro={val_f1_macro:.4f}")
        print(classification_report(y_val, y_val_pred, digits=4))

        # refit final em train + val
        final_pipeline = clone(clf_cfg["pipeline"])
        final_pipeline.set_params(**grid.best_params_)
        final_pipeline.fit(X_train_val, y_train_val)

        # teste
        y_test_pred = final_pipeline.predict(X_test)
        test_acc = accuracy_score(y_test, y_test_pred)
        test_f1_macro = f1_score(y_test, y_test_pred, average="macro")

        print(f"\n[TEST] {experiment_name} | Acc={test_acc:.4f} | F1-macro={test_f1_macro:.4f}")
        print(classification_report(y_test, y_test_pred, digits=4))

        results.append(
            {
                "model": experiment_name,
                "embedding_model": encoder_name,
                "classifier": clf_name,
                "cv_best_f1_macro": grid.best_score_,
                "val_acc": val_acc,
                "val_f1_macro": val_f1_macro,
                "test_acc": test_acc,
                "test_f1_macro": test_f1_macro,
                "best_params": str(grid.best_params_),
            }
        )

        for i in range(len(meta_val)):
            rows_per_item.append(
                {
                    "split": "val",
                    "embedding_model": encoder_name,
                    "classifier": clf_name,
                    "model": experiment_name,
                    "idx": meta_val.iloc[i]["idx"],
                    "dataset_id": meta_val.iloc[i]["dataset_id"],
                    "true_label_id": int(y_val[i]),
                    "pred_label_id": int(y_val_pred[i]),
                }
            )

        for i in range(len(meta_test)):
            rows_per_item.append(
                {
                    "split": "test",
                    "embedding_model": encoder_name,
                    "classifier": clf_name,
                    "model": experiment_name,
                    "idx": meta_test.iloc[i]["idx"],
                    "dataset_id": meta_test.iloc[i]["dataset_id"],
                    "true_label_id": int(y_test[i]),
                    "pred_label_id": int(y_test_pred[i]),
                }
            )

# ============================================================
# SALVA RESULTADOS
# ============================================================

results_df = pd.DataFrame(results).sort_values(
    by=["test_f1_macro", "val_f1_macro"],
    ascending=False,
)

pred_df = pd.DataFrame(rows_per_item)

results_path = MODEL_DIR / "classical_models_results.csv"
val_pred_path = MODEL_DIR / "classical_models_val_predictions.csv"
test_pred_path = MODEL_DIR / "classical_models_test_predictions.csv"

results_df.to_csv(results_path, index=False)
pred_df[pred_df["split"] == "val"].to_csv(val_pred_path, index=False)
pred_df[pred_df["split"] == "test"].to_csv(test_pred_path, index=False)

print("\n" + "=" * 100)
print("RESUMO FINAL")
print("=" * 100)
print(results_df.to_string(index=False))

print("\n[OK] Arquivos salvos:")
print(" -", results_path)
print(" -", val_pred_path)
print(" -", test_pred_path)


####################################################################################################
CARREGANDO EMBEDDINGS SALVOS PARA: bertimbau_base
####################################################################################################

----------------------------------------------------------------------------------------------------
TREINANDO: bertimbau_base__logreg
----------------------------------------------------------------------------------------------------
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Melhores hiperparâmetros: {'clf__C': 0.1, 'clf__solver': 'liblinear'}
Melhor CV F1-macro: 0.845734601032938

[VAL] bertimbau_base__logreg | Acc=0.8628 | F1-macro=0.8469
              precision    recall  f1-score   support

           0     0.8818    0.9111    0.8962      4084
           1     0.8239    0.7729    0.7976      2197

    accuracy                         0.8628      6281
   macro avg     0.8528    0.8420    0.8469      6281
weighted 